# 보험 데이터 추가 수집 (파이프라인 통일용)

**이 노트북이 하는 일**: 통일 파이프라인에 넣을 보험 데이터를 **공통 형식 CSV**로 만들어 저장합니다. 분석은 하지 않습니다. 분석은 통일 파이프라인(공통 코드)에서 합니다. IT의 `it_data_collect.ipynb`와 같은 구조·컬럼입니다.

**왜 다시 받나요?** 통일 규칙대로 계산하려면 기존 파일럿보다 더 옛날 데이터가 필요합니다.

| 쓰는 곳 | 필요한 데이터 |
|---|---|
| 회사 기준 점수 (직전 8분기) | 2022년 초 발표의 "평소"를 계산하려면 2020~2021년 실적과 주가 |
| 대조군 판정 | 2022년 1월 발표의 대조군을 고르려면 2021년 말 발표 기록 |
| 평소 거래량, 플라시보 | 2021년 중반부터의 주가 |

**기존 보험 파일럿에서 바뀐 점**

| 항목 | 기존 | 이번 |
|---|---|---|
| 실적 시작 | 2021Q1 | **2020년 1월 발표부터** (Alpha Vantage 다시 받기) |
| 주가 시작 | 2020-06 | **2020-01** |
| 주가 끝 | 2026-09-30 | **2026-10-20** (IT와 같은 날짜) |
| 종가 | `close`, `adjclose` 둘 다 있음 | 그대로 |
| 시장 기준 / 업종 ETF | SPY / KIE | 그대로 |
| 검증값 | 7개사 × 2025Q1~2026Q2를 SEC 8-K로 확인 | 그대로 덮어씀 (발표 시각·EPS) |
| AIG 실적 | Alpha Vantage | **야후 값으로 교체** (AV에 회계상 EPS와 틀린 장 전/후가 들어 있었음, 7~8번 참고) |
| 출처 확인 | 없음 | **야후와 비교해서 엇갈린 분기를 `check_flag`에 표시** (강건성용) |

**회사 목록**은 기존 보험 파일럿과 같은 미국 손해보험 23개사입니다.

**저장되는 파일** (`data/insurance/` 폴더)

| 파일 | 한 줄 = | 컬럼 |
|---|---|---|
| `companies_insurance.csv` | 회사 1개 | `ticker, name, subgroup` |
| `earnings_insurance.csv` | 실적발표 1번 | `ticker, announce_datetime, timing, eps_actual, eps_consensus` (+ 참고용 `fiscal_end, source, check_flag`) |
| `prices_insurance.csv` | 종목 1개 × 거래일 1일 | `date, ticker, close, adjclose, volume` |
| `split_check_insurance.csv` | 분할 1번 | 분할 확인 결과 (공유용) |
| `raw_av_earnings.csv` | 실적발표 1번 | Alpha Vantage 원본 (다시 받지 않으려고 저장) |
| `eps_check_insurance.csv` | 실적발표 1번 | Alpha Vantage와 야후 실적 비교표 (공유용) |

## 1. 준비

첫 칸에서 필요한 패키지를 설치합니다.

코랩에서 결과를 드라이브에 남기려면 `USE_DRIVE = True`로 바꾸세요. 그러면 `DRIVE_FOLDER` 안에 `data/insurance/`가 만들어집니다.

**Alpha Vantage API 키**는 실행하면 입력창이 뜹니다. 키를 코드에 직접 적지 마세요 (깃허브에 올라가면 다른 사람이 쓸 수 있습니다). 무료 키는 하루 25번까지라서 23개사를 한 번에 받을 수 있습니다. 한 번 받으면 `raw_av_earnings.csv`로 저장되고, 다음부터는 다시 받지 않습니다.

In [23]:
!pip install yfinance requests

USE_DRIVE = False
DRIVE_FOLDER = "/content/drive/MyDrive/did_pilot"

import os, time
from getpass import getpass
from pathlib import Path
import numpy as np
import pandas as pd
import requests
import yfinance as yf

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_FOLDER, exist_ok=True)
    os.chdir(DRIVE_FOLDER)

OUT = Path("data/insurance")
OUT.mkdir(parents=True, exist_ok=True)

START = "2020-01-01"     # 주가·실적 시작
END = "2026-10-21"       # 이 날짜는 포함되지 않아서 실제로는 2026-10-20까지 (IT와 같음)
ETFS = ["SPY", "KIE"]    # 시장 기준(SPY), 보험 업종 ETF(KIE)
VERIFY_FILE = OUT / "pc_panel_2025Q1_2026Q2.csv"   # SEC 8-K로 확인한 검증값 (기존 파일럿 파일)
print("yfinance", yf.__version__, "/ 저장 위치:", OUT.resolve())

yfinance 0.2.66 / 저장 위치: /content/data/insurance


## 2. 회사 목록 (기존 보험 파일럿 23개사 고정)

`subgroup`은 보험 안의 세부 묶음입니다. 대조군을 "같은 하위그룹 / 다른 하위그룹"으로 나눠보는 강건성 분석에 씁니다.
- `large_multiline`: 대형 다각화 (기업보험 + 개인보험)
- `personal_lines`: 개인보험(자동차·주택) 중심
- `specialty_reins`: 특수·기업보험·재보험

참고: Everest Group은 2023년에 티커가 `RE` → `EG`로 바뀌었습니다. 야후·Alpha Vantage 모두 `EG`로 옛날 기록까지 나옵니다.

In [24]:
COMPANIES = [
    ("TRV", "Travelers", "large_multiline"), ("CB", "Chubb", "large_multiline"),
    ("AIG", "American International Group", "large_multiline"), ("HIG", "Hartford Financial", "large_multiline"),
    ("CNA", "CNA Financial", "large_multiline"), ("CINF", "Cincinnati Financial", "large_multiline"),
    ("ALL", "Allstate", "personal_lines"), ("MCY", "Mercury General", "personal_lines"),
    ("KMPR", "Kemper", "personal_lines"), ("HCI", "HCI Group", "personal_lines"),
    ("WRB", "W. R. Berkley", "specialty_reins"), ("MKL", "Markel", "specialty_reins"),
    ("ACGL", "Arch Capital", "specialty_reins"), ("AXS", "Axis Capital", "specialty_reins"),
    ("KNSL", "Kinsale Capital", "specialty_reins"), ("RLI", "RLI Corp", "specialty_reins"),
    ("SIGI", "Selective Insurance", "specialty_reins"), ("AFG", "American Financial Group", "specialty_reins"),
    ("THG", "Hanover Insurance", "specialty_reins"), ("ORI", "Old Republic", "specialty_reins"),
    ("PLMR", "Palomar", "specialty_reins"), ("RNR", "RenaissanceRe", "specialty_reins"),
    ("EG", "Everest Group", "specialty_reins"),
]
companies = pd.DataFrame(COMPANIES, columns=["ticker", "name", "subgroup"])
tickers = companies["ticker"].tolist()
companies.to_csv(OUT / "companies_insurance.csv", index=False)
companies["subgroup"].value_counts()

,count
subgroup,
specialty_reins,13
large_multiline,6
personal_lines,4


## 3. 주가·거래량

IT와 똑같이 `auto_adjust=False`로 받아서 두 종류의 종가를 다 저장합니다.
- `close`: 주식 분할만 반영한 종가. SUE 분모용
- `adjclose`: 분할과 배당을 모두 반영한 종가. 수익률 계산용

In [25]:
raw = yf.download(tickers + ETFS, start=START, end=END, auto_adjust=False, progress=False)

parts = []
for col, name in [("Close", "close"), ("Adj Close", "adjclose"), ("Volume", "volume")]:
    s = raw[col].reset_index().melt(id_vars="Date", var_name="ticker", value_name=name)
    parts.append(s.set_index(["Date", "ticker"]))
prices = (pd.concat(parts, axis=1).reset_index()
          .rename(columns={"Date": "date"})
          .dropna(subset=["close"])
          .sort_values(["ticker", "date"]).reset_index(drop=True))
prices["date"] = pd.to_datetime(prices["date"]).dt.strftime("%Y-%m-%d")
prices.to_csv(OUT / "prices_insurance.csv", index=False)

print(f"주가 {len(prices):,}줄 / 종목 {prices['ticker'].nunique()}개 (회사 {len(tickers)} + ETF {len(ETFS)})")
print("전체 기간:", prices["date"].min(), "~", prices["date"].max())

주가 42,725줄 / 종목 25개 (회사 23 + ETF 2)
전체 기간: 2020-01-02 ~ 2026-10-20


**주가 점검**: 종목별 첫날과 마지막 날입니다.
- 모두 `2020-01-02`에 시작해야 정상입니다 (23개사 모두 2020년 전에 상장).
- 마지막 날이 `2026-10-20` 근처가 아니면 데이터가 덜 받아진 것이니 다시 실행하세요.
- 빠진 종목이 있으면 야후가 일시적으로 막은 것일 수 있으니 몇 분 뒤 다시 실행하세요.

In [26]:
span = prices.groupby("ticker")["date"].agg(first="min", last="max", n_days="count")
missing = sorted(set(tickers + ETFS) - set(span.index))
print("아예 안 받아진 종목:", missing or "없음")
print("늦게 시작하는 종목:", span[span["first"] > "2020-01-10"].index.tolist() or "없음")
print("마지막 날이 2026-10-15보다 이른 종목:", span[span["last"] < "2026-10-15"].index.tolist() or "없음")
print("거래일 수가 다른 종목:", span[span["n_days"] != span["n_days"].mode()[0]].index.tolist() or "없음")
span

아예 안 받아진 종목: 없음
늦게 시작하는 종목: 없음
마지막 날이 2026-10-15보다 이른 종목: 없음
거래일 수가 다른 종목: 없음


,first,last,n_days
ticker,,,
ACGL,2020-01-02,2026-10-20,1709
AFG,2020-01-02,2026-10-20,1709
AIG,2020-01-02,2026-10-20,1709
ALL,2020-01-02,2026-10-20,1709
AXS,2020-01-02,2026-10-20,1709
CB,2020-01-02,2026-10-20,1709
CINF,2020-01-02,2026-10-20,1709
CNA,2020-01-02,2026-10-20,1709
EG,2020-01-02,2026-10-20,1709


## 4. 실적 (Alpha Vantage, 2020년부터)

기존 보험 파일럿과 같은 출처(Alpha Vantage `EARNINGS`)입니다. 회사마다 상장 이후 모든 분기를 한 번에 주니까, 그중 **2020년 1월 이후 발표**만 남깁니다.

**발표 시각**: Alpha Vantage는 정확한 시각 대신 장 전/장 후만 줍니다.
- `pre-market` → `BMO` (장 시작 전)
- `post-market` → `AMC` (장 마감 후)
- 그 외 → `unknown` (통일 규칙상 분석에서 제외 예정)

`announce_datetime`의 시각은 공통 형식을 맞추려고 넣은 **대표값**입니다 (BMO 08:00, AMC 16:05, 모름 00:00). 반응일 계산에는 `timing`만 쓰입니다. SEC로 확인한 분기는 다음 칸에서 실제 접수 시각으로 바뀝니다.

무료 키는 1분에 5번 제한이 있어서 한 회사마다 13초씩 쉽니다 (23개사 약 5분).

In [27]:
RAW_AV = OUT / "raw_av_earnings.csv"

def fetch_av(ticker, key, tries=3):
    for _ in range(tries):
        js = requests.get("https://www.alphavantage.co/query",
                          params={"function": "EARNINGS", "symbol": ticker, "apikey": key}, timeout=30).json()
        if "quarterlyEarnings" in js:
            d = pd.DataFrame(js["quarterlyEarnings"])
            d["ticker"] = ticker
            return d
        msg = str(js.get("Note") or js.get("Information") or js)[:150]
        if "per day" in msg or "daily" in msg:
            raise RuntimeError("하루 요청 한도 초과: 내일 다시 실행하면 이어서 받습니다. " + msg)
        print(f"  {ticker}: 잠깐 막힘 → 60초 쉬고 다시 ({msg})")
        time.sleep(60)
    raise RuntimeError(f"{ticker}: 세 번 시도했는데 실패")

av = pd.read_csv(RAW_AV, dtype=str) if RAW_AV.exists() else pd.DataFrame(columns=["ticker"])
todo = [t for t in tickers if t not in set(av["ticker"])]
if todo:
    AV_KEY = os.environ.get("ALPHAVANTAGE_KEY") or getpass("Alpha Vantage API 키 입력: ")
    for t in todo:
        try:
            av = pd.concat([av, fetch_av(t, AV_KEY).astype(str)], ignore_index=True)
            av.to_csv(RAW_AV, index=False)          # 한 회사 받을 때마다 저장 (중간에 끊겨도 이어서 받기)
            print(t, "OK")
        except RuntimeError as e:
            print(e); break
        time.sleep(13)
print(f"Alpha Vantage 원본: {av['ticker'].nunique()}/{len(tickers)}개사, {len(av)}줄",
      "/ 아직 못 받은 회사:", [t for t in tickers if t not in set(av['ticker'])] or "없음")

Alpha Vantage 원본: 23/23개사, 2574줄 / 아직 못 받은 회사: 없음


In [28]:
e = av.rename(columns={"fiscalDateEnding": "fiscal_end", "reportedDate": "announce_date",
                       "reportTime": "report_time", "reportedEPS": "eps_actual", "estimatedEPS": "eps_consensus"})
e["eps_actual"] = pd.to_numeric(e["eps_actual"], errors="coerce")       # 'None' 같은 값은 빈칸으로
e["eps_consensus"] = pd.to_numeric(e["eps_consensus"], errors="coerce")
e["timing"] = e["report_time"].map({"pre-market": "BMO", "post-market": "AMC"}).fillna("unknown")
e = e[e["eps_actual"].notna() & (e["announce_date"] >= START) & (e["announce_date"] < END)].copy()

REP_TIME = {"BMO": "08:00", "AMC": "16:05", "unknown": "00:00"}   # 대표 시각 (위 설명 참고)
def to_dt(date, hhmm):
    return pd.Timestamp(f"{date} {hhmm}").tz_localize("America/New_York").strftime("%Y-%m-%d %H:%M:%S%z")

e["announce_datetime"] = [to_dt(d, REP_TIME[t]) for d, t in zip(e["announce_date"], e["timing"])]
e["source"] = "AlphaVantage"
fq = pd.to_datetime(e["fiscal_end"])
e["quarter"] = fq.dt.year.astype(str) + "Q" + fq.dt.quarter.astype(str)
print(f"2020년 이후 실적 {len(e)}건 / 발표 시각:", e["timing"].value_counts().to_dict())

2020년 이후 실적 625건 / 발표 시각: {'AMC': 518, 'BMO': 107}


## 5. SEC 검증값 덮어쓰기 (기존 파일럿과 같음)

기존 파일럿에서 7개사 × 2025Q1~2026Q2를 SEC 8-K 접수 시각과 보도자료 원문으로 확인했습니다. 이 분기는 Alpha Vantage 값 대신 확인한 값을 씁니다.
- 발표 날짜·시각 → SEC 8-K 접수 시각
- 실제 EPS → 보도자료의 조정 EPS (Alpha Vantage에 GAAP EPS가 들어 있던 분기가 있었음)

`pc_panel_2025Q1_2026Q2.csv`를 `data/insurance/` 폴더에 넣고 실행하세요. 파일이 없으면 이 단계는 건너뜁니다.

In [29]:
if VERIFY_FILE.exists():
    ver = pd.read_csv(VERIFY_FILE, dtype={"edgar_time_et": str})
    ver = ver[["quarter", "ticker", "announce_date", "edgar_time_et", "timing", "reported_eps"]].rename(
        columns={"announce_date": "v_date", "edgar_time_et": "v_time", "timing": "v_timing", "reported_eps": "v_eps"})
    e = e.merge(ver, on=["quarter", "ticker"], how="left")
    hit = e["v_date"].notna()
    eps_changed = int((hit & (e["v_eps"] - e["eps_actual"]).abs().gt(0.005)).sum())
    date_changed = int((hit & (e["v_date"] != e["announce_date"])).sum())
    e.loc[hit, "announce_date"] = e.loc[hit, "v_date"]
    e.loc[hit, "timing"] = e.loc[hit, "v_timing"]
    e.loc[hit, "eps_actual"] = e.loc[hit, "v_eps"]
    e.loc[hit, "announce_datetime"] = [to_dt(d, (t if isinstance(t, str) and ":" in t else REP_TIME[tm]))
                                       for d, t, tm in zip(e.loc[hit, "v_date"], e.loc[hit, "v_time"], e.loc[hit, "v_timing"])]
    e.loc[hit, "source"] = "AV + SEC/보도자료 검증"
    e = e.drop(columns=["v_date", "v_time", "v_timing", "v_eps"])
    print(f"검증값으로 덮어쓴 분기 {int(hit.sum())}건 (EPS 바뀜 {eps_changed}건, 발표일 바뀜 {date_changed}건)")
else:
    print("검증 파일 없음 → Alpha Vantage 값 그대로 사용:", VERIFY_FILE)

earn = (e[["ticker", "announce_datetime", "timing", "eps_actual", "eps_consensus", "fiscal_end", "source"]]
        .drop_duplicates(subset=["ticker", "announce_datetime"])
        .sort_values(["ticker", "announce_datetime"]).reset_index(drop=True))
earn.to_csv(OUT / "earnings_insurance.csv", index=False)
print(f"저장: 실적 {len(earn)}건, {earn['ticker'].nunique()}개사")

검증값으로 덮어쓴 분기 42건 (EPS 바뀜 5건, 발표일 바뀜 0건)
저장: 실적 625건, 23개사


**실적 점검**: 2020년 1월부터 지금까지면 회사당 **약 27분기**가 정상입니다.
- `first`가 2020년 초가 아니면, 그 회사는 2022년 초 발표의 "직전 8분기"가 모자랄 수 있습니다.
- `consensus_missing`이 0보다 크면 예상 EPS가 빈 분기가 있다는 뜻입니다.
- `unknown_timing`이 0보다 크면 장 전/후를 모르는 분기입니다. IR 보도자료로 확인하면 살릴 수 있습니다.

아래 **"크게 빗나간 분기"**는 실제 EPS와 예상의 차이가 예상치 자체보다 큰 분기입니다 (예: 예상 1.00인데 실제 2.10이나 −0.20). 보험은 재해 손실로 진짜 크게 빗나갈 수 있지만, Alpha Vantage가 조정 EPS 대신 GAAP EPS를 넣은 경우일 수도 있습니다 (기존 파일럿에서 5건 발견). 의심되면 보도자료로 확인하세요.

In [30]:
ed = pd.to_datetime(earn["announce_datetime"].str[:10])
cov = (earn.assign(d=ed.dt.strftime("%Y-%m-%d"))
       .groupby("ticker").agg(n=("d", "size"), first=("d", "min"), last=("d", "max"),
                              consensus_missing=("eps_consensus", lambda s: int(s.isna().sum())),
                              unknown_timing=("timing", lambda s: int((s == "unknown").sum()))))
print(f"회사당 평균 {cov['n'].mean():.1f}분기, 최소 {cov['n'].min()}분기")
print("2020-04-30 이후에 시작하는 회사:", cov[cov["first"] > "2020-04-30"].index.tolist() or "없음")

gap = (earn["eps_actual"] - earn["eps_consensus"]).abs() / earn["eps_consensus"].abs()
big = earn[(gap > 1) & (earn["eps_actual"] - earn["eps_consensus"]).abs().gt(0.3)]
print(f"\n크게 빗나간 분기 {len(big)}건 (확인 후보):")
print(big[["ticker", "announce_datetime", "eps_consensus", "eps_actual", "source"]].to_string(index=False))
cov.sort_values("n")

회사당 평균 27.2분기, 최소 27분기
2020-04-30 이후에 시작하는 회사: 없음

크게 빗나간 분기 48건 (확인 후보):
ticker        announce_datetime  eps_consensus  eps_actual           source
  ACGL 2021-10-27 16:05:00-0400           0.35      0.7400     AlphaVantage
   AIG 2020-05-04 08:00:00-0400           0.72      1.9800     AlphaVantage
   AIG 2020-08-03 08:00:00-0400           0.50     -9.1500     AlphaVantage
   AIG 2021-02-16 08:00:00-0500           0.93     -0.0605     AlphaVantage
   AIG 2021-05-06 08:00:00-0400           0.97      4.4100     AlphaVantage
   AIG 2021-11-04 08:00:00-0400           0.88      1.9200     AlphaVantage
   AIG 2022-02-16 08:00:00-0500           1.19      4.3800     AlphaVantage
   AIG 2022-05-03 08:00:00-0400           1.18      5.1500     AlphaVantage
   AIG 2022-08-08 08:00:00-0400           1.10      3.7800     AlphaVantage
   AIG 2022-11-01 08:00:00-0400           0.53      3.5000     AlphaVantage
   AIG 2023-05-04 08:00:00-0400           1.42     -0.1200     AlphaVantage
   AIG 2023-11

,n,first,last,consensus_missing,unknown_timing
ticker,,,,,
ACGL,27,2020-02-11,2026-07-28,0,0
AFG,27,2020-02-03,2026-08-04,0,0
AIG,27,2020-02-13,2026-08-06,0,0
ALL,27,2020-02-04,2026-08-05,0,0
AXS,27,2020-01-29,2026-07-27,0,0
CINF,27,2020-02-05,2026-07-27,0,0
CNA,27,2020-02-10,2026-08-03,0,0
EG,27,2020-02-10,2026-07-29,0,0
KNSL,27,2020-02-20,2026-07-23,0,0


## 6. 확인: 주식 분할한 회사의 EPS가 분할 조정돼 있나요?

SUE는 (실제 EPS − 예상 EPS) ÷ 주가(`close`)입니다. `close`는 분할이 반영돼 있으니까 **EPS도 분할이 반영돼 있어야** 단위가 맞습니다. (IT 노트북과 같은 방법)

**읽는 법**: 분할 날짜 앞뒤로
- EPS가 비슷한 크기로 이어지면 → **분할 조정돼 있음 (정상)**
- EPS가 분할 비율만큼 갑자기 뛰거나 떨어지면 → **조정 안 됨**

보험사 중에는 W. R. Berkley(WRB)가 3대2 분할을 여러 번 했습니다. 보험 EPS는 재해 때문에 분기마다 크게 흔들려서 자동 판정이 틀릴 수 있으니, 애매하면 표를 직접 보세요.

In [31]:
split_rows = []
for t in tickers:
    sp = yf.Ticker(t).splits
    if sp is None or len(sp) == 0:
        continue
    sp.index = pd.to_datetime(sp.index, utc=True)
    sp = sp[sp.index >= pd.Timestamp(START, tz="UTC")]
    for when, ratio in sp.items():
        et = earn[earn.ticker == t].assign(dt=pd.to_datetime(earn.loc[earn.ticker == t, "announce_datetime"], utc=True))
        before = et[et.dt < when].tail(4)["eps_actual"].mean()
        after = et[et.dt >= when].head(4)["eps_actual"].mean()
        r = before / after if after else np.nan
        verdict = ("조정 안 된 듯 (확인 필요)" if pd.notna(r) and abs(r - ratio) < abs(r - 1) else "조정된 듯 (정상)")
        split_rows.append(dict(ticker=t, split_date=when.date(), ratio=ratio,
                               eps_before_4q=round(before, 3), eps_after_4q=round(after, 3), verdict=verdict))
        print(f"\n{t}: {when.date()} {ratio:g}대1 분할")
        print(et[(et.dt >= when - pd.Timedelta(days=400)) & (et.dt <= when + pd.Timedelta(days=400))]
              [["announce_datetime", "eps_consensus", "eps_actual"]].to_string(index=False))

split_check = pd.DataFrame(split_rows, columns=["ticker", "split_date", "ratio", "eps_before_4q", "eps_after_4q", "verdict"])
split_check.to_csv(OUT / "split_check_insurance.csv", index=False)
split_check if len(split_check) else print("2020년 이후 분할한 회사 없음")


WRB: 2022-03-24 1.5대1 분할
       announce_datetime  eps_consensus  eps_actual
2021-04-20 16:05:00-0400           0.36        0.48
2021-07-22 16:05:00-0400           0.45        0.52
2021-10-21 16:05:00-0400           0.42        0.59
2022-01-27 16:05:00-0500           0.56        0.68
2022-04-25 16:05:00-0400           0.63        0.73
2022-07-21 16:05:00-0400           0.57        0.75
2022-10-24 16:05:00-0400           0.57        0.67
2023-01-26 16:05:00-0500           0.73        0.77
2023-04-20 16:05:00-0400           0.82        0.67

WRB: 2024-07-11 1.5대1 분할
       announce_datetime  eps_consensus  eps_actual
2023-07-20 16:05:00-0400           0.72        0.76
2023-10-23 16:05:00-0400           0.78        0.90
2024-01-24 16:05:00-0500           0.92        0.97
2024-04-23 08:00:00-0400           0.95        1.04
2024-07-22 16:05:00-0400           0.93        1.04
2024-10-21 16:05:00-0400           0.96        0.91
2025-01-27 16:05:00-0500           0.95        1.13
2025-04-21 1

,ticker,split_date,ratio,eps_before_4q,eps_after_4q,verdict
0,WRB,2022-03-24,1.5,0.568,0.730,조정된 듯 (정상)
1,WRB,2024-07-11,1.5,0.918,1.022,조정된 듯 (정상)
2,RLI,2025-01-16,2.0,0.810,0.750,조정된 듯 (정상)


## 7. 야후 실적과 비교 (Alpha Vantage 값 확인용)

4번의 "크게 빗나간 분기"가 진짜 서프라이즈인지 데이터 오류인지 가리려고, 같은 분기를 야후(`get_earnings_dates`, IT와 같은 방법)에서도 받아 비교합니다.
- 실제 EPS가 0.05 넘게 다르면 `eps_mismatch`
- 장 전/후가 다르면 `timing_mismatch`

**결과 요약** (624/625 분기 짝지음)
- **AIG**: 17분기 EPS, 15분기 장 전/후가 다름. AV 값이 예상과 동떨어진 회계상 EPS였고(예: 2020-08 예상 0.50 / AV −9.15 / 야후 0.66), AV는 장 전으로 적었지만 SEC로 확인한 AIG 발표는 모두 장 마감 후 → **야후가 맞음**
- **나머지**: 야후가 늘 맞지는 않음. SEC로 확인한 분기에서는 오히려 야후가 회계상 EPS를 넣은 경우가 있었음 (ALL 2025-07·11, 2026-02 / WRB 2025-10 / TRV 2026-01 / AIG 2025-08)
- 비교표는 `eps_check_insurance.csv`로 저장

In [33]:
def get_yf(t):
    d = yf.Ticker(t).get_earnings_dates(limit=40)
    if d is None or len(d) == 0:
        return pd.DataFrame()
    d = d.reset_index()
    ren = {d.columns[0]: "yf_dt"}
    for c in d.columns[1:]:
        lc = str(c).lower()
        if "reported" in lc: ren[c] = "yf_actual"
        elif "estimate" in lc: ren[c] = "yf_consensus"
    d = d.rename(columns=ren)
    ts = pd.to_datetime(d["yf_dt"], utc=True).dt.tz_convert("America/New_York")
    h = ts.dt.hour + ts.dt.minute / 60
    d["yf_timing"] = np.select([h == 0, h < 9.5, h >= 16], ["unknown", "BMO", "AMC"], "during")
    d["yf_date"] = ts.dt.tz_localize(None).dt.normalize()
    d["ticker"] = t
    return d[["ticker", "yf_date", "yf_timing", "yf_actual", "yf_consensus"]].dropna(subset=["yf_actual"])

parts = []
for t in tickers:
    try:
        parts.append(get_yf(t))
    except Exception as ex:
        print(t, "야후 실패:", repr(ex)[:80])
    time.sleep(1.0)
yfe = pd.concat(parts, ignore_index=True)

a = earn.assign(date=pd.to_datetime(earn["announce_datetime"].str[:10])).sort_values("date")
cmp = pd.merge_asof(a, yfe.sort_values("yf_date"), left_on="date", right_on="yf_date",
                    by="ticker", direction="nearest", tolerance=pd.Timedelta(days=3))
cmp["eps_diff"] = (cmp["eps_actual"] - cmp["yf_actual"]).abs()
cmp["eps_mismatch"] = cmp["eps_diff"] > 0.05
cmp["timing_mismatch"] = cmp["yf_timing"].isin(["BMO", "AMC"]) & (cmp["timing"] != cmp["yf_timing"])
cmp = cmp.sort_values(["ticker", "date"])
cmp.drop(columns=["date"]).to_csv(OUT / "eps_check_insurance.csv", index=False)

m = cmp["yf_actual"].notna()
print(f"야후와 짝지어진 분기 {m.sum()}/{len(cmp)}건")
print(f"실제 EPS가 0.05 넘게 다른 분기 {int(cmp.eps_mismatch.sum())}건 / 장 전·후가 다른 분기 {int(cmp.timing_mismatch.sum())}건")
print("\n회사별 (EPS 다름, 장 전·후 다름):")
print(cmp.groupby("ticker")[["eps_mismatch", "timing_mismatch"]].sum().query("eps_mismatch > 0 or timing_mismatch > 0").astype(int).to_string())
print("\nEPS가 다른 분기:")
cmp[cmp.eps_mismatch][["ticker", "announce_datetime", "eps_consensus", "eps_actual", "yf_consensus", "yf_actual", "source"]]

야후와 짝지어진 분기 624/625건
실제 EPS가 0.05 넘게 다른 분기 36건 / 장 전·후가 다른 분기 20건

회사별 (EPS 다름, 장 전·후 다름):
        eps_mismatch  timing_mismatch
ticker                               
ACGL               2                0
AIG               17               15
ALL                3                0
EG                 1                0
HCI                2                2
KNSL               1                2
MKL                1                0
ORI                2                1
PLMR               2                0
RNR                2                0
TRV                2                0
WRB                1                0

EPS가 다른 분기:


,ticker,announce_datetime,eps_consensus,eps_actual,yf_consensus,yf_actual,source
452,ACGL,2024-10-30 16:05:00-0400,2.10,2.5600,1.95,1.99,AlphaVantage
536,ACGL,2025-10-27 16:05:00-0400,2.25,2.7700,2.43,3.56,AlphaVantage
36,AIG,2020-05-04 08:00:00-0400,0.72,1.9800,0.82,0.11,AlphaVantage
63,AIG,2020-08-03 08:00:00-0400,0.50,-9.1500,0.56,0.66,AlphaVantage
89,AIG,2020-11-05 08:00:00-0500,0.54,0.3200,0.55,0.81,AlphaVantage
111,AIG,2021-02-16 08:00:00-0500,0.93,-0.0605,0.93,0.94,AlphaVantage
137,AIG,2021-05-06 08:00:00-0400,0.97,4.4100,0.98,1.05,AlphaVantage
160,AIG,2021-08-05 08:00:00-0400,1.20,0.1100,1.20,1.52,AlphaVantage
182,AIG,2021-11-04 08:00:00-0400,0.88,1.9200,0.89,0.97,AlphaVantage
204,AIG,2022-02-16 08:00:00-0500,1.19,4.3800,1.18,1.58,AlphaVantage


## 8. 정리: AIG만 야후 값으로 교체, 나머지는 표시만

- **AIG** (SEC 확인 분기 제외): 실제 EPS·장 전/후·날짜를 야후 값으로 바꾸고 `source = "AV→야후 수정 (AIG)"`
- **나머지 회사에서 엇갈린 분기**: 어느 쪽이 맞는지 모르니 값은 그대로 두고 `check_flag`에 표시 → 공통 파이프라인에서 **이 분기를 빼고 돌려도 결과가 같은지** 강건성으로 확인
- **SEC로 확인한 분기**: 확인한 값이 우선이라 그대로

결과: AV 567건, SEC 확인 42건, AIG 수정 16건 / 표시된 분기 19건 (EPS 14, 장 전·후 5)

In [34]:
# 8. 정리: AIG는 야후 값으로 교체, 나머지 불일치는 표시만
c = cmp.drop(columns=["date"]).copy()
verified = c["source"].str.contains("SEC")
fix = (c["ticker"] == "AIG") & ~verified & c["yf_actual"].notna()

# AIG: 실제 EPS·장 전후·날짜를 야후 값으로
c.loc[fix & c.eps_mismatch, "eps_actual"] = c.loc[fix & c.eps_mismatch, "yf_actual"]
t_fix = fix & c.timing_mismatch
c.loc[t_fix, "timing"] = c.loc[t_fix, "yf_timing"]
c.loc[t_fix, "announce_datetime"] = [to_dt(d.strftime("%Y-%m-%d"), REP_TIME[t])
                                     for d, t in zip(c.loc[t_fix, "yf_date"], c.loc[t_fix, "timing"])]
c.loc[fix & (c.eps_mismatch | c.timing_mismatch), "source"] = "AV→야후 수정 (AIG)"

# 나머지: 바꾸지 않고 표시만 (강건성 분석에서 빼보기용)
other = ~fix & ~verified
c["check_flag"] = np.select([other & c.eps_mismatch & c.timing_mismatch, other & c.eps_mismatch, other & c.timing_mismatch],
                            ["EPS·장전후 출처 불일치", "EPS 출처 불일치", "장전후 출처 불일치"], "")

earn = (c[["ticker", "announce_datetime", "timing", "eps_actual", "eps_consensus", "fiscal_end", "source", "check_flag"]]
        .sort_values(["ticker", "announce_datetime"]).reset_index(drop=True))
earn.to_csv(OUT / "earnings_insurance.csv", index=False)

print("출처별:", earn["source"].value_counts().to_dict())
print("표시된 분기:", earn["check_flag"].value_counts().drop("", errors="ignore").to_dict())
print("AIG 장 전·후:", earn[earn.ticker == "AIG"]["timing"].value_counts().to_dict())
earn[earn.check_flag != ""][["ticker", "announce_datetime", "timing", "eps_consensus", "eps_actual", "check_flag"]]

출처별: {'AlphaVantage': 567, 'AV + SEC/보도자료 검증': 42, 'AV→야후 수정 (AIG)': 16}
표시된 분기: {'EPS 출처 불일치': 14, '장전후 출처 불일치': 5}
AIG 장 전·후: {'AMC': 26, 'BMO': 1}


,ticker,announce_datetime,timing,eps_consensus,eps_actual,check_flag
19,ACGL,2024-10-30 16:05:00-0400,AMC,2.1000,2.56,EPS 출처 불일치
23,ACGL,2025-10-27 16:05:00-0400,AMC,2.2500,2.77,EPS 출처 불일치
240,EG,2025-10-27 16:05:00-0400,AMC,14.2500,6.09,EPS 출처 불일치
259,HCI,2023-11-07 16:05:00-0500,AMC,0.3100,1.41,EPS 출처 불일치
260,HCI,2024-03-07 16:05:00-0500,AMC,1.4100,3.22,EPS 출처 불일치
264,HCI,2025-02-27 08:00:00-0500,BMO,-2.4100,0.31,장전후 출처 불일치
268,HCI,2026-02-25 08:00:00-0500,BMO,4.9100,7.25,장전후 출처 불일치
345,KNSL,2025-02-13 08:00:00-0500,BMO,4.3217,4.62,장전후 출처 불일치
348,KNSL,2025-10-23 16:05:00-0400,AMC,4.8200,5.21,EPS 출처 불일치
349,KNSL,2026-02-12 08:00:00-0500,BMO,5.1800,5.81,장전후 출처 불일치


## 9. 마무리 체크리스트

1. `data/insurance/` 안에 아래 파일이 있는지 확인
   - `companies_insurance.csv`, `earnings_insurance.csv`, `prices_insurance.csv` (공통 파이프라인 입력)
   - `split_check_insurance.csv` (분할 확인 결과, 공유용)
   - `raw_av_earnings.csv`, `pc_panel_2025Q1_2026Q2.csv` (원본·검증값, 재현용)
   - `eps_check_insurance.csv` (야후 비교표, 공유용)
2. 파일들을 레포의 `data/insurance/`에, 이 노트북을 `notebook/`에 올리기
3. 공통 파이프라인 담당에게 컬럼 이름 공유
   - 실적: `ticker, announce_datetime, timing, eps_actual, eps_consensus` (IT와 같음, 뒤에 참고용 `fiscal_end, source, check_flag`)
   - 주가: `date, ticker, close, adjclose, volume` (SPY, KIE 포함)
   - 보험 `announce_datetime`의 시각은 대표값이고, SEC 확인 분기만 실제 시각이라는 점
4. 한 줄 공유: 분할 3건 모두 정상 / AIG는 야후 값으로 교체 / 출처가 엇갈린 19건은 `check_flag` 표시 (강건성에서 빼고 돌려보기)

In [32]:
for f in ["companies_insurance.csv", "earnings_insurance.csv", "prices_insurance.csv",
          "split_check_insurance.csv", "raw_av_earnings.csv", "pc_panel_2025Q1_2026Q2.csv"]:
    p = OUT / f
    print(f"OK  {p}  ({p.stat().st_size/1024:,.0f} KB)" if p.exists() else f"없음 {p}")

OK  data/insurance/companies_insurance.csv  (1 KB)
OK  data/insurance/earnings_insurance.csv  (42 KB)
OK  data/insurance/prices_insurance.csv  (2,438 KB)
OK  data/insurance/split_check_insurance.csv  (0 KB)
OK  data/insurance/raw_av_earnings.csv  (151 KB)
OK  data/insurance/pc_panel_2025Q1_2026Q2.csv  (6 KB)
